## Phase 3: Apply Mapping to Both DataFrames

In [ ]:
import pandas as pd
import os

In [ ]:
off_clean = pd.read_csv("off_data_clean2.csv") # insert your right off dataset csv
usda_clean = pd.read_csv("usda_data_clean2.csv") # insert your right usda dataset csv

In [ ]:
# ── Step 8: Apply the mapping to both dataframes ─────────────────────────
# Load mapping (so this cell works even if you restart the kernel)
cat_map = pd.read_csv("category_mapping.csv")
l2_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l2"]))
l1_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l1"]))

# Map L1 and L2 onto both dataframes; L3 = original cat
off_clean["cat_l1"] = off_clean["cat"].map(l1_lookup)
off_clean["cat_l2"] = off_clean["cat"].map(l2_lookup)
off_clean["cat_l3"] = off_clean["cat"]  # finest grain

usda_clean["cat_l1"] = usda_clean["cat"].map(l1_lookup)
usda_clean["cat_l2"] = usda_clean["cat"].map(l2_lookup)
usda_clean["cat_l3"] = usda_clean["cat"]

print("=== OFF coverage ===")
print(f"  L1 mapped: {off_clean['cat_l1'].notna().sum():,} / {len(off_clean):,} ({off_clean['cat_l1'].notna().mean()*100:.1f}%)")
print(f"  L2 mapped: {off_clean['cat_l2'].notna().sum():,} / {len(off_clean):,}")
print(f"  L1 unmapped: {off_clean['cat_l1'].isna().sum():,} rows")

print("\n=== USDA coverage ===")
print(f"  L1 mapped: {usda_clean['cat_l1'].notna().sum():,} / {len(usda_clean):,} ({usda_clean['cat_l1'].notna().mean()*100:.1f}%)")
print(f"  L2 mapped: {usda_clean['cat_l2'].notna().sum():,} / {len(usda_clean):,}")
print(f"  L1 unmapped: {usda_clean['cat_l1'].isna().sum():,} rows")

=== OFF coverage ===
  L1 mapped: 37,397 / 40,996 (91.2%)
  L2 mapped: 37,397 / 40,996
  L1 unmapped: 3,599 rows

=== USDA coverage ===
  L1 mapped: 1,826,482 / 1,826,573 (100.0%)
  L2 mapped: 1,826,482 / 1,826,573
  L1 unmapped: 91 rows


In [ ]:
# ── Step 9: Handle unmapped rows (tiny categories) ───────────────────────
# For rows where cat wasn't in our mapping (the <3 item categories),
# classify them by item_name using the same LLM function, or assign "other"

unmapped_off = off_clean[off_clean["cat_l2"].isna()]
unmapped_usda = usda_clean[usda_clean["cat_l2"].isna()]

print(f"Unmapped OFF rows: {len(unmapped_off)} ({len(unmapped_off)/len(off_clean)*100:.1f}%)")
print(f"Unmapped USDA rows: {len(unmapped_usda)} ({len(unmapped_usda)/len(usda_clean)*100:.1f}%)")

# Option A: assign "other" (simple, fast)
off_clean["cat_l1"].fillna("other", inplace=True)
off_clean["cat_l2"].fillna("other", inplace=True)
usda_clean["cat_l1"].fillna("other", inplace=True)
usda_clean["cat_l2"].fillna("other", inplace=True)

print("\nAfter filling unmapped → 'other':")
print(f"  OFF L1 unique: {off_clean['cat_l1'].nunique()}")
print(f"  OFF L2 unique: {off_clean['cat_l2'].nunique()}")
print(f"  USDA L1 unique: {usda_clean['cat_l1'].nunique()}")
print(f"  USDA L2 unique: {usda_clean['cat_l2'].nunique()}")

Unmapped OFF rows: 3599 (8.8%)
Unmapped USDA rows: 91 (0.0%)

After filling unmapped → 'other':
  OFF L1 unique: 20
  OFF L2 unique: 86
  USDA L1 unique: 20
  USDA L2 unique: 86


/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_2067/949075583.py:12: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  off_clean["cat_l1"].fillna("other", inplace=True)
/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_2067/949075583.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Seri

In [ ]:
# ── Step 10: Validate — spot check a few L1 groups ───────────────────────
combined = pd.concat([off_clean, usda_clean], ignore_index=True)

for l1 in ["dairy & eggs", "fruits", "snacks", "prepared & frozen meals", "beverages"]:
    subset = combined[combined["cat_l1"] == l1]
    sample = subset.sample(min(5, len(subset)), random_state=42)
    print(f"\n{'='*60}")
    print(f"L1: {l1}  ({len(subset):,} rows)")
    print(f"{'='*60}")
    print(sample[["item_name", "cat_l1", "cat_l2", "cat_l3", "source"]].to_string(index=False))


L1: dairy & eggs  (252,840 rows)
                                           item_name       cat_l1                    cat_l2                    cat_l3 source
            southern- style custard, southern- style dairy & eggs            dairy desserts       puddings & custards   usda
          no added sugar pint coffee ice cream, pint dairy & eggs ice cream & frozen yogurt ice cream & frozen yogurt   usda
                      plain whole milk yogurt, plain dairy & eggs                    yogurt                    yogurt   usda
            vanilla greek whole milk yogurt, vanilla dairy & eggs                    yogurt                    yogurt   usda
dierbergs kitchen, holiday dessert spread, cranberry dairy & eggs                    cheese                    cheese   usda

L1: fruits  (53,351 rows)
                                                  item_name cat_l1                    cat_l2                                          cat_l3 source
                       cold-pressed juice

In [ ]:
# ── Step 11: Save final datasets ─────────────────────────────────────────
off_clean.to_csv("off_data_ontology.csv", index=False)
usda_clean.to_csv("usda_data_ontology.csv", index=False)

print(f"Saved off_data_ontology.csv  ({len(off_clean):,} rows)")
print(f"Saved usda_data_ontology.csv ({len(usda_clean):,} rows)")
print(f"\nColumns: {off_clean.columns.tolist()}")

Saved off_data_ontology.csv  (40,996 rows)
Saved usda_data_ontology.csv (1,826,573 rows)

Columns: ['item_name', 'cat', 'kcal_100g', 'fat_100g', 'carbs_100g', 'protein_100g', 'source', 'cat_l1', 'cat_l2', 'cat_l3']
